# LoRA fine-tuning of DeepSeek-Coder-1.3B-Instruct on xLAM function calling

LoRA-only version of the project plan:

| Plan step | Where |
|---|---|
| 1. DeepSeek-Coder-1.3B-Instruct as base model | Config |
| 2. One prompt template (request + function definitions -> expected call) | Section 3 |
| 3. Reserve unseen APIs; split the rest into train / val / seen-tool test | Section 3 |
| 4. Fixed seed, checkpoint, sequence length, effective batch size, final epochs | Config |
| 5. Preliminary runs to choose hyperparameters (train / val only) | Section 4 |
| 6. Final LoRA fine-tune on the full training split | Section 5 |
| Evaluation: seen-tool and unseen-tool test sets, base vs LoRA | Section 6 |

**Before you start**
1. Runtime -> Change runtime type -> GPU (a T4 works for QUICK_TEST; use an L4/A100 for a full run).
2. Open the [dataset page](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k), log in, and **accept the terms** (gated).
3. Have a Hugging Face token ready.

**Keep `QUICK_TEST = True` for your first run** to check that every cell works on small subsets. Set it to `False` for the real experiment.

## 0. Install

In [1]:
!pip install -q -U transformers datasets peft trl accelerate
!pip uninstall -y torchao
# Then: Runtime -> Restart session, and continue from the next cell (do NOT re-run this one).

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


## 1. Log in to Hugging Face

In [1]:
from huggingface_hub import notebook_login
notebook_login()

## 2. Config

Everything that must stay identical across fine-tuning methods (plan step 4) lives here: model, seed, sequence length, effective batch size, final epochs.

In [7]:
   del trainer
   import gc, torch
   gc.collect(); torch.cuda.empty_cache()

In [8]:
import json, os, random, gc, math, time
import torch, pandas as pd
from google.colab import drive

drive.mount("/content/drive")
OUTPUT_DIR = "/content/drive/MyDrive/xlam-lora-deepseek"     # splits, configs, checkpoints, adapter
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- fixed across all methods ----
SEED          = 42
MODEL         = "deepseek-ai/deepseek-coder-1.3b-instruct"
MAX_LEN       = 1536            # tokens; longer examples are dropped (never truncated)
PER_DEVICE_BS = 2
GRAD_ACCUM    = 8               # effective batch size = 2 * 8 = 16
EPOCHS_FINAL  = 2
UNSEEN_FRAC   = 0.05            # fraction of API (function) names reserved for the unseen-tool test

# ---- set False for the real experiment ----
QUICK_TEST = True

if QUICK_TEST:
    MAX_TRAIN, VAL_N, HP_TRAIN, HP_VAL, N_TEST = 600, 100, 300, 100, 40
    LRS, RANKS = [1e-4, 2e-4], [8]
else:
    MAX_TRAIN, VAL_N, HP_TRAIN, HP_VAL, N_TEST = None, 1000, 2000, 300, 300
    LRS, RANKS = [1e-4, 2e-4, 4e-4], [8, 16]

assert torch.cuda.is_available(), "Switch the runtime to a GPU."
USE_BF16 = torch.cuda.get_device_capability()[0] >= 8      # False on T4 (uses fp16), True on L4/A100
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(torch.cuda.get_device_name(0), "| bf16:", USE_BF16)

random.seed(SEED); torch.manual_seed(SEED)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tesla T4 | bf16: False


## 3. Data: prompt template and splits

**Prompt template (step 2).** System instruction + available function definitions + user query as the prompt; the expected function call(s) as the completion. The model's own chat template wraps it, and the loss is computed on the completion only.

**Splits (step 3).** The dataset has no API-ID field, so an API's identity is its **function name**:
1. A random 5% of all function names are *reserved*.
2. Examples whose answer calls a reserved function -> **unseen-tool pool**.
3. Examples that merely list a reserved function as a distractor -> **dropped**, so training never sees a reserved schema.
4. The remaining examples are shuffled and split 80/10/10 into **train / val / seen-tool test**.

Function names are only a proxy: the same name can appear with slightly different schemas. Examples longer than `MAX_LEN` tokens are dropped.

In [9]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

SYSTEM = (
    "You are a function-calling assistant. Given a user query and a list of "
    "tools, respond ONLY with a JSON list of tool calls, each of the form "
    '{"name": ..., "arguments": {...}}.'
)

def build_messages(row):
    tools = json.loads(row["tools"])
    return [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Tools:\n{json.dumps(tools)}\n\nQuery: {row['query']}"},
    ]

def to_prompt_completion(row):
    answers = json.loads(row["answers"])
    return {
        "prompt": build_messages(row),
        "completion": [{"role": "assistant", "content": json.dumps(answers)}],
    }

def make_splits(tools_json, answers_json, seed, unseen_frac, val_frac=0.1, test_frac=0.1):
    tools_of = [{t["name"] for t in json.loads(s)} for s in tools_json]
    gold_of  = [{a["name"] for a in json.loads(s)} for s in answers_json]
    all_names = sorted(set().union(*tools_of))
    rng = random.Random(seed)
    reserved = set(rng.sample(all_names, max(1, int(unseen_frac * len(all_names)))))
    unseen, pool, dropped = [], [], 0
    for i, (t, g) in enumerate(zip(tools_of, gold_of)):
        if g & reserved:
            unseen.append(i)
        elif t & reserved:
            dropped += 1
        else:
            pool.append(i)
    rng.shuffle(pool)
    n_val, n_test = int(val_frac * len(pool)), int(test_frac * len(pool))
    return dict(reserved=sorted(reserved), unseen=unseen, dropped=dropped,
                val=pool[:n_val], seen_test=pool[n_val:n_val + n_test], train=pool[n_val + n_test:])

raw = load_dataset("Salesforce/xlam-function-calling-60k", split="train")
raw = raw.add_column("idx", list(range(len(raw))))
S = make_splits(raw["tools"], raw["answers"], SEED, UNSEEN_FRAC)
print(f"reserved APIs: {len(S['reserved'])} | unseen pool: {len(S['unseen'])} | dropped (distractor only): {S['dropped']}")
print(f"train pool: {len(S['train'])} | val pool: {len(S['val'])} | seen-test pool: {len(S['seen_test'])}")

def fits(row):
    msgs = build_messages(row) + [{"role": "assistant", "content": json.dumps(json.loads(row["answers"]))}]
    text = tok.apply_chat_template(msgs, tokenize=False)
    return len(tok(text, add_special_tokens=False).input_ids) <= MAX_LEN

def make_split(idx_list, limit=None):
    if limit:
        idx_list = idx_list[: int(limit * 1.3)]          # slack for rows dropped by the length filter
    ds = raw.select(idx_list).filter(fits)
    if limit and len(ds) > limit:
        ds = ds.select(range(limit))
    return ds

train_raw     = make_split(S["train"], MAX_TRAIN)
val_raw       = make_split(S["val"], VAL_N)
seen_test_raw = make_split(S["seen_test"], N_TEST)
unseen_raw    = make_split(S["unseen"], N_TEST)

train_ds = train_raw.map(to_prompt_completion, remove_columns=train_raw.column_names)
val_ds   = val_raw.map(to_prompt_completion, remove_columns=val_raw.column_names)
hp_train = train_ds.select(range(min(HP_TRAIN, len(train_ds))))
hp_val   = val_ds.select(range(min(HP_VAL, len(val_ds))))
print(f"used -> train {len(train_ds)} | val {len(val_ds)} | seen-test {len(seen_test_raw)} | unseen-test {len(unseen_raw)}")

# Sanity check: see exactly what the model is trained on
ex = train_ds[0]
print("\n----- rendered training example (first 1200 chars) -----")
print(tok.apply_chat_template(ex["prompt"] + ex["completion"], tokenize=False)[:1200])

# Save the split membership so the same splits can be reused for other fine-tuning methods
with open(f"{OUTPUT_DIR}/splits.json", "w") as f:
    json.dump({"seed": SEED, "unseen_frac": UNSEEN_FRAC, "reserved_apis": S["reserved"],
               "train": list(train_raw["idx"]), "val": list(val_raw["idx"]),
               "seen_test": list(seen_test_raw["idx"]), "unseen_test": list(unseen_raw["idx"])}, f)

reserved APIs: 180 | unseen pool: 3893 | dropped (distractor only): 3981
train pool: 41702 | val pool: 5212 | seen-test pool: 5212
used -> train 600 | val 100 | seen-test 40 | unseen-test 40

----- rendered training example (first 1200 chars) -----
<｜begin▁of▁sentence｜>You are a function-calling assistant. Given a user query and a list of tools, respond ONLY with a JSON list of tool calls, each of the form {"name": ..., "arguments": {...}}.### Instruction:
Tools:
[{"name": "std_deviation", "description": "Calculates the standard deviation of a list of numbers.", "parameters": {"numbers": {"description": "The list of numbers.", "type": "List[float]"}}}, {"name": "permutation_count", "description": "Calculates the number of permutations of k elements from a set of n elements.", "parameters": {"n": {"description": "The total number of elements in the set.", "type": "int"}, "k": {"description": "The number of elements to choose for the permutation.", "type": "int"}}}]

Query: Find the stan

## 4. Preliminary runs: choose hyperparameters (train + val only)

Short LoRA runs (1 epoch on a training subset), scored by **validation loss on the completion tokens**. The test sets are not touched. The grid is learning rate x LoRA rank (`lora_alpha = 2 * r`). Each run reloads the base model so runs are independent.

These runs use a subset and 1 epoch, so the winning learning rate is a good guide for the full run but not guaranteed to be optimal there.

In [10]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

def make_trainer(lr, r, train, evalset, epochs, out_dir, final=False):
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE, device_map={"": 0})
    steps_per_epoch = math.ceil(len(train) / (PER_DEVICE_BS * GRAD_ACCUM))
    total_steps = steps_per_epoch * epochs
    args = SFTConfig(
        output_dir=out_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=PER_DEVICE_BS,
        per_device_eval_batch_size=PER_DEVICE_BS,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=lr,
        lr_scheduler_type="cosine",
        warmup_steps=max(1, int(0.03 * total_steps)),
        max_length=MAX_LEN,
        bf16=USE_BF16, fp16=not USE_BF16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        seed=SEED, data_seed=SEED,
        logging_steps=10,
        eval_strategy="epoch" if final else "no",
        save_strategy="epoch" if final else "no",
        save_total_limit=2,
        report_to="none",
    )
    peft_config = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05,
                             target_modules="all-linear", task_type="CAUSAL_LM")
    return SFTTrainer(model=model, args=args, train_dataset=train, eval_dataset=evalset,
                      processing_class=tok, peft_config=peft_config)

hp_results = []
for lr in LRS:
    for r in RANKS:
        t0 = time.time()
        trainer = make_trainer(lr, r, hp_train, hp_val, epochs=1, out_dir="hp-tmp")
        trainer.train()
        m = trainer.evaluate()
        hp_results.append({"lr": lr, "r": r, "val_loss": round(m["eval_loss"], 4),
                           "minutes": round((time.time() - t0) / 60, 1)})
        print(hp_results[-1])
        del trainer
        gc.collect(); torch.cuda.empty_cache()

hp_table = pd.DataFrame(hp_results).sort_values("val_loss").reset_index(drop=True)
display(hp_table)
best = hp_table.iloc[0].to_dict()
print("Selected:", best)
hp_table.to_json(f"{OUTPUT_DIR}/hp_search.json", orient="records", indent=2)

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Step,Training Loss
10,0.132357


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
0.132357,0.041805,19,0.026026,0.988376,136009.000000


{'lr': 0.0001, 'r': 8, 'val_loss': 0.0418, 'minutes': 3.0}


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Step,Training Loss
10,0.116010


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
0.116010,0.033934,19,0.026825,0.989544,136009.000000


{'lr': 0.0002, 'r': 8, 'val_loss': 0.0339, 'minutes': 2.8}


,lr,r,val_loss,minutes
0,0.0002,8,0.0339,2.8
1,0.0001,8,0.0418,3.0


Selected: {'lr': 0.0002, 'r': 8.0, 'val_loss': 0.0339, 'minutes': 2.8}


## 5. Final LoRA fine-tune on the full training split

Uses the selected learning rate and rank, the full training split (`MAX_TRAIN = None` when `QUICK_TEST = False`), and `EPOCHS_FINAL` epochs. No early stopping, so the model after the last epoch is the one evaluated. Checkpoints go to Drive each epoch; if the session disconnects, rerun cells up to the trainer creation and use `trainer.train(resume_from_checkpoint=True)`.

In [11]:
FINAL_DIR   = f"{OUTPUT_DIR}/final"
ADAPTER_DIR = f"{OUTPUT_DIR}/final-adapter"

trainer = make_trainer(best["lr"], int(best["r"]), train_ds, val_ds, EPOCHS_FINAL, FINAL_DIR, final=True)
trainer.model.print_trainable_parameters()

# After a disconnect, use this instead:  trainer.train(resume_from_checkpoint=True)
trainer.train()

trainer.save_model(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)

with open(f"{OUTPUT_DIR}/run_config.json", "w") as f:
    json.dump({"method": "lora", "model": MODEL, "seed": SEED, "max_len": MAX_LEN,
               "effective_batch_size": PER_DEVICE_BS * GRAD_ACCUM, "epochs": EPOCHS_FINAL,
               "lr": best["lr"], "r": int(best["r"]), "lora_alpha": 2 * int(best["r"]),
               "n_train": len(train_ds), "n_val": len(val_ds), "quick_test": QUICK_TEST}, f, indent=2)
print("Saved adapter to", ADAPTER_DIR)

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

... (2 outputs left out)

Dropping fully masked examples from eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

trainable params: 7,495,680 || all params: 1,353,967,616 || trainable%: 0.5536


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.082056,0.025996,0.028176,0.992479,279011.000000
2,0.029688,0.025139,0.024406,0.992826,558022.000000


Saved adapter to /content/drive/MyDrive/xlam-lora-deepseek/final-adapter


## 6. Evaluation: seen-tool and unseen-tool test sets

Five metrics (valid-call rate, function accuracy, argument accuracy, schema compliance, complete-call accuracy, plus exact match), computed for the **base model** (adapter off) and the **LoRA model** on both test sets.

In [12]:
import json, math, re
from collections import Counter
from tqdm.auto import tqdm
import pandas as pd

# ---------------------------------------------------------------- parsing
def parse_calls(text):
    """Strict parse. Returns a list of {"name": str, "arguments": dict}, or None if the format is invalid."""
    try:
        data = json.loads(text.strip())
    except Exception:
        return None
    if not isinstance(data, list):
        return None
    for c in data:
        if not (isinstance(c, dict) and isinstance(c.get("name"), str)
                and isinstance(c.get("arguments"), dict)):
            return None
    return data

# ----------------------------------------------------------- type checking
def type_ok(value, type_str):
    """Check a JSON value against an xLAM-style type string ('int', 'List[str]', 'Optional[float]', 'str, optional'...).
    Unknown types are accepted."""
    tl = (type_str or "").strip().lower()
    optional = "optional" in tl
    if value is None:
        return optional
    m = re.match(r"optional\[(.*)\]$", tl)
    if m:
        tl = m.group(1).strip()
    elif "[" not in tl and "," in tl:          # e.g. "int, optional"
        tl = tl.split(",")[0].strip()
    base = tl.split("[")[0].strip()

    if base in ("str", "string", "text"):
        return isinstance(value, str)
    if base in ("int", "integer"):
        return isinstance(value, int) and not isinstance(value, bool)
    if base in ("float", "number", "double"):
        return isinstance(value, (int, float)) and not isinstance(value, bool)
    if base in ("bool", "boolean"):
        return isinstance(value, bool)
    if base in ("list", "array", "tuple", "set", "sequence"):
        if not isinstance(value, list):
            return False
        if "[" in tl and tl.endswith("]"):
            inner = tl[tl.index("[") + 1:-1]
            if "," not in inner:                # only check elements for simple List[x]
                return all(type_ok(v, inner) for v in value)
        return True
    if base in ("dict", "object", "mapping"):
        return isinstance(value, dict)
    return True

# ------------------------------------------------------ value comparison
def norm(v):
    """Lenient normalisation: case/whitespace-insensitive strings, '5' == 5 == 5.0."""
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        s = v.strip().lower()
        try:
            f = float(s)
            return f if math.isfinite(f) else s
        except ValueError:
            return s
    if isinstance(v, list):
        return [norm(x) for x in v]
    if isinstance(v, dict):
        return {k: norm(x) for k, x in v.items()}
    return v

def match_calls(gold, pred):
    """Pair each gold call with an unused predicted call of the same name (best argument overlap)."""
    used, pairs = set(), []
    for g in gold:
        best, best_sc = None, -1
        for i, p in enumerate(pred):
            if i in used or p["name"] != g["name"]:
                continue
            sc = sum(1 for k, v in g["arguments"].items()
                     if k in p["arguments"] and norm(p["arguments"][k]) == norm(v))
            if sc > best_sc:
                best, best_sc = i, sc
        if best is not None:
            used.add(best)
        pairs.append((g, pred[best] if best is not None else None))
    return pairs

# ------------------------------------------------------------ one example
def score_example(row, text):
    tools = json.loads(row["tools"])
    gold = json.loads(row["answers"])
    schema = {t["name"]: (t.get("parameters") or {}) for t in tools}

    r = dict(valid=0, func=0, schema=0, complete=0, exact=0,
             args_correct=0, args_total=sum(len(g["arguments"]) for g in gold))
    pred = parse_calls(text)
    if pred is None:
        return r
    r["valid"] = 1

    # 2) function accuracy: same multiset of function names (order-insensitive)
    r["func"] = int(Counter(p["name"] for p in pred) == Counter(g["name"] for g in gold))

    # 4) schema compliance: known function, known parameter names, right types
    ok = True
    for p in pred:
        params = schema.get(p["name"])
        if params is None:
            ok = False
            break
        for k, v in p["arguments"].items():
            spec = params.get(k)
            if spec is None or not type_ok(v, spec.get("type") if isinstance(spec, dict) else None):
                ok = False
                break
        if not ok:
            break
    r["schema"] = int(ok)

    # 3) argument accuracy + 5) complete-call accuracy
    all_complete, all_exact = True, True
    for g, p in match_calls(gold, pred):
        if p is None:
            all_complete = all_exact = False
            continue
        correct = [k for k, v in g["arguments"].items()
                   if k in p["arguments"] and norm(p["arguments"][k]) == norm(v)]
        r["args_correct"] += len(correct)
        # Gold calls contain every required argument, so "all gold args present with the
        # right value" == "all required arguments correct". Extra args don't fail this check
        # (they fail schema compliance / exact match instead).
        gold_ok = len(correct) == len(g["arguments"])
        if not gold_ok:
            all_complete = False
        if not (gold_ok and set(p["arguments"]) == set(g["arguments"])):
            all_exact = False
    r["complete"] = int(bool(r["func"]) and all_complete)
    r["exact"] = int(bool(r["func"]) and all_exact)
    return r

# ------------------------------------------------------------- aggregate
def compute_metrics(rows, texts):
    res = [score_example(r, t) for r, t in zip(rows, texts)]
    n = len(res)
    s = lambda k: sum(x[k] for x in res)
    return {
        "Valid-call rate":        100 * s("valid") / n,
        "Function accuracy":      100 * s("func") / n,
        "Argument accuracy":      100 * s("args_correct") / max(1, s("args_total")),
        "Schema compliance":      100 * s("schema") / n,
        "Complete-call accuracy": 100 * s("complete") / n,
        "Exact match (extra)":    100 * s("exact") / n,
    }



In [13]:
from tqdm.auto import tqdm

model = trainer.model
model.eval()

def generate(messages, max_new_tokens=512):
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             use_cache=True, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def run_model(rows):
    return [generate(build_messages(r)) for r in tqdm(rows)]

columns, saved = {}, {}
for split_name, rows in [("seen", list(seen_test_raw)), ("unseen", list(unseen_raw))]:
    with model.disable_adapter():                       # base model, LoRA switched off
        base_texts = run_model(rows)
    tuned_texts = run_model(rows)                       # LoRA model
    columns[f"{split_name} | base"] = compute_metrics(rows, base_texts)
    columns[f"{split_name} | LoRA"] = compute_metrics(rows, tuned_texts)
    saved[split_name] = {"gold": [r["answers"] for r in rows], "base": base_texts, "lora": tuned_texts}

results = pd.DataFrame(columns).round(1)
display(results)

results.to_csv(f"{OUTPUT_DIR}/eval_results.csv")
with open(f"{OUTPUT_DIR}/predictions.json", "w") as f:
    json.dump(saved, f)

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

,seen | base,seen | LoRA,unseen | base,unseen | LoRA
Valid-call rate,0.0,100.0,0.0,95.0
Function accuracy,0.0,95.0,0.0,92.5
Argument accuracy,0.0,83.5,0.0,82.8
Schema compliance,0.0,97.5,0.0,92.5
Complete-call accuracy,0.0,80.0,0.0,75.0
Exact match (extra),0.0,77.5,0.0,72.5
